# 01 — Macro analysis of the OpenAlex DH corpus

Goal: characterise the full collection and its biases (publication years, work types, OA status, language, country, venues, concepts) and produce a per-year frequency series for AI-related terms in **abstracts**, alongside `machine learning` / `deep learning` comparators.

Inputs:
- `openalex_data/openalex_dh_works.json` (~470 MB, streamed with `ijson`).

Outputs:
- `outputs/macro/works_metadata.parquet` — flat metadata DataFrame.
- `outputs/macro/term_counts_by_year.parquet` — per-year AI / ML / DL counts.
- Inline figures (bias profile + diachronic curves).


In [ ]:
from __future__ import annotations

# Auto-reload edits to lib/* without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
import importlib
from pathlib import Path

# allow importing analysis_code/lib from the notebook
ROOT = Path.cwd()
while ROOT.name and not (ROOT / "analysis_code").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "analysis_code"))

import ijson
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Force-reload in case the kernel already imported a stale version.
import lib, lib.ai_lexicon, lib.text_clean
importlib.reload(lib.ai_lexicon)
importlib.reload(lib.text_clean)

from lib import (
    OPENALEX_JSON, OUT_MACRO,
)
from lib.ai_lexicon import (
    AI_GROUPS,
    classify_text,
    compile_comparator_regex,
)
from lib.text_clean import reconstruct_abstract_from_inverted_index

sns.set_theme(context="notebook", style="whitegrid")
print("openalex json:", OPENALEX_JSON, OPENALEX_JSON.exists())
print("size MB:", round(OPENALEX_JSON.stat().st_size / 1e6, 1))


## 1. Stream-parse JSON → flat DataFrame

In [ ]:
def _flatten_work(w: dict) -> dict:
    pl = w.get("primary_location") or {}
    src = (pl.get("source") or {}) if isinstance(pl, dict) else {}
    oa = w.get("open_access") or {}
    auths = w.get("authorships") or []
    countries: list[str] = []
    for a in auths:
        for c in a.get("countries", []) or []:
            countries.append(c)
    primary_topic = (w.get("primary_topic") or {})
    field = (primary_topic.get("field") or {}).get("display_name") if primary_topic else None
    subfield = (primary_topic.get("subfield") or {}).get("display_name") if primary_topic else None

    abstract = w.get("abstract")
    if not abstract:
        abstract = reconstruct_abstract_from_inverted_index(w.get("abstract_inverted_index"))

    wid = (w.get("id") or "").rsplit("/", 1)[-1]
    return {
        "id": wid,
        "doi": w.get("doi"),
        "title": w.get("title") or w.get("display_name"),
        "publication_year": w.get("publication_year"),
        "publication_date": w.get("publication_date"),
        "type": w.get("type"),
        "language": w.get("language"),
        "is_oa": oa.get("is_oa"),
        "oa_status": oa.get("oa_status"),
        "venue": src.get("display_name"),
        "venue_id": src.get("id"),
        "venue_type": src.get("type"),
        "host_organization": src.get("host_organization_name"),
        "n_authors": len(auths),
        "first_country": countries[0] if countries else None,
        "all_countries": "|".join(sorted(set(countries))) if countries else None,
        "primary_topic": primary_topic.get("display_name") if primary_topic else None,
        "primary_field": field,
        "primary_subfield": subfield,
        "concepts": "|".join(
            (c.get("display_name") or "")
            for c in (w.get("concepts") or [])[:5]
        ),
        "cited_by_count": w.get("cited_by_count"),
        "has_fulltext": w.get("has_fulltext"),
        "is_retracted": w.get("is_retracted"),
        "is_paratext": w.get("is_paratext"),
        "abstract": abstract,
    }


rows: list[dict] = []
with open(OPENALEX_JSON, "rb") as f:
    for w in tqdm(ijson.items(f, "item"), desc="works"):
        rows.append(_flatten_work(w))
df = pd.DataFrame(rows)
print(df.shape)
df.head(3)


In [ ]:
# Quick sanity
assert df["id"].is_unique, "duplicate work IDs in JSON"
print("n works:", len(df))
print("n with abstract:", df["abstract"].notna().sum())
print("year range:", int(df["publication_year"].min()), "→", int(df["publication_year"].max()))

out_meta = OUT_MACRO / "works_metadata.parquet"
df.to_parquet(out_meta, index=False)
print("saved", out_meta)


## 1b. Pre-processing filter

Restrict the working DataFrame to:
* abstracts in **English** (`language == "en"`),
* work types other than `peer-review`, `editorial`, `reference-entry`,
  `erratum`, and `standard` (none are primary research outputs), and
* exclude works hosted by **Zenodo** (CERN European Organization for Nuclear
  Research) from the venue side, which inflates the venue ranking with
  generic deposit records.

downstream cells operate on the filtered `df`.
The unfiltered DataFrame stays available as `df_raw` for reference; all

In [ ]:
df_raw = df.copy()
n0 = len(df_raw)

DROP_TYPES = {"peer-review", "editorial", "reference-entry", "erratum", "standard"}
DROP_HOSTS = {"Zenodo (CERN European Organization for Nuclear Research)"}

mask_en        = df_raw["language"].eq("en")
mask_not_type  = ~df_raw["type"].isin(DROP_TYPES)
mask_not_host  = ~df_raw["venue"].isin(DROP_HOSTS) & ~df_raw["venue"].eq("Zenodo")
df = df_raw[mask_en & mask_not_type & mask_not_host].reset_index(drop=True)

print(f"raw works:                      {n0}")
print(f"dropped non-English:            {(~mask_en).sum()}")
print(f"dropped excluded work types:    {(mask_en & ~mask_not_type).sum()}")
print(f"  type counts dropped:          {df_raw.loc[mask_en & ~mask_not_type, 'type'].value_counts().to_dict()}")
print(f"dropped Zenodo / CERN venue:    {(mask_en & mask_not_type & ~mask_not_host).sum()}")
print(f"kept after all filters:         {len(df)}  ({len(df)/n0:.1%})")

print(f"  of which with abstract:       {df['abstract'].notna().sum()}")



out_meta_f = OUT_MACRO / "works_metadata_filtered.parquet"
print("saved", out_meta_f)
df.to_parquet(out_meta_f, index=False)

## 2. Bias profile

In [ ]:
def _bar(series: pd.Series, title: str, top: int | None = 15, ax=None):
    s = series.value_counts(dropna=False).head(top) if top else series.value_counts(dropna=False)
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, max(3, 0.35 * len(s))))
    s.iloc[::-1].plot(kind="barh", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("works")
    return ax


# Year distribution
fig, ax = plt.subplots(figsize=(10, 3.5))
df["publication_year"].dropna().astype(int).value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Works per publication year")
ax.set_xlabel("year"); ax.set_ylabel("works")
plt.tight_layout(); plt.show()


In [ ]:
len(df[df["type"]=="book-chapter"])

In [ ]:
df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
_bar(df["type"], "Work type", ax=axes[0, 0])
_bar(df["oa_status"], "Open access status", ax=axes[0, 1])
_bar(df["language"], "Language", ax=axes[1, 0])
_bar(df["first_country"], "First-author country (top 15)", top=15, ax=axes[1, 1])
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
_bar(df["venue"], "Top venues", top=20, ax=axes[0])
_bar(df["primary_field"], "Primary OpenAlex field", top=15, ax=axes[1])
plt.tight_layout(); plt.show()


In [ ]:
# show all the instances of the top concepts
df["concepts"].dropna().str.split("|").explode().tolist().count("History")

In [ ]:
# Top concepts (flatten the 'concepts' pipe-joined column)
concepts = (
    df["concepts"].dropna().str.split("|").explode().str.strip()
)
concepts = concepts[concepts != ""]
fig, ax = plt.subplots(figsize=(8, 6))
concepts.value_counts().head(20).iloc[::-1].plot(kind="barh", ax=ax)
ax.set_title("Top 20 OpenAlex concepts (across top-5 per work)")
ax.set_xlabel("works")
plt.tight_layout(); plt.show()


### 2.1 Geographical distribution of first-author country

Choropleth of the first-author country (`first_country`, ISO-2 from
OpenAlex authorships). Counts are works per country; the colour scale is
log-stretched so smaller producers remain visible next to the long-tail
leaders.

In [ ]:
import numpy as np
import pycountry
import plotly.express as px

# ISO-2 -> ISO-3 (plotly choropleth wants ISO-3 alpha codes).
_iso2_to_iso3 = {c.alpha_2: c.alpha_3 for c in pycountry.countries}

geo = (
    df["first_country"].dropna().astype(str).str.upper()
    .value_counts().rename_axis("iso2").reset_index(name="works")
)
geo["iso3"] = geo["iso2"].map(_iso2_to_iso3)
geo["country"] = geo["iso2"].map(
    lambda x: (pycountry.countries.get(alpha_2=x).name
               if pycountry.countries.get(alpha_2=x) else x)
)
unmapped = geo[geo["iso3"].isna()]
if not unmapped.empty:
    print("WARN unmapped ISO-2 codes (dropped from map):",
          unmapped[["iso2", "works"]].to_dict("records"))
geo = geo.dropna(subset=["iso3"])
geo["log_works"] = np.log10(geo["works"])

fig = px.choropleth(
    geo,
    locations="iso3",
    color="log_works",
    hover_name="country",
    hover_data={"iso3": False, "log_works": False, "works": True},
    color_continuous_scale="Viridis",
    projection="natural earth",
    title="First-author country (works per country, log-scaled colour)",
)
# Replace log-scale tick labels with real work counts.
tick_exps = list(range(0, int(np.ceil(geo["log_works"].max())) + 1))
fig.update_coloraxes(
    colorbar=dict(
        title="works",
        tickvals=tick_exps,
        ticktext=[f"{10**e:,}" for e in tick_exps],
    )
)
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0), height=520)
fig.show()


In [ ]:
geo

In [ ]:
# Static top-20 companion bar (uses full country names for readability).
top20 = geo.sort_values("works", ascending=False).head(20)
fig, ax = plt.subplots(figsize=(8, 6))
top20.iloc[::-1].plot(kind="barh", x="country", y="works", legend=False, ax=ax)
ax.set_title("Top 20 first-author countries")
ax.set_xlabel("works"); ax.set_ylabel("")
plt.tight_layout(); plt.show()


In [ ]:
# Side-by-side: top first-author countries  |  works per publication year
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import cm

TOP_N = 20

# --- left panel: top countries ---------------------------------------------
top_countries = (
    geo.sort_values("works", ascending=False).head(TOP_N).iloc[::-1]
)

# --- right panel: works per year -------------------------------------------
years = (
    df["publication_year"].dropna().astype(int)
    .value_counts().sort_index()
)

# --- figure -----------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "semibold",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

fig, (ax_l, ax_r) = plt.subplots(
    1, 2, figsize=(15, 7),
    gridspec_kw={"width_ratios": [1, 1.15], "wspace": 0.28},
)
fig.patch.set_facecolor("white")

# Left: horizontal bars, viridis gradient by rank
colors_l = cm.viridis(np.linspace(0.15, 0.85, len(top_countries)))
bars = ax_l.barh(top_countries["country"], top_countries["works"],
                 color=colors_l, edgecolor="white", linewidth=0.6)
ax_l.set_title(f"Top {TOP_N} first-author countries", loc="left", pad=12)
ax_l.set_xlabel("works")
ax_l.tick_params(axis="y", length=0)
ax_l.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax_l.grid(axis="x", linestyle=":", alpha=0.5)
ax_l.set_axisbelow(True)
xmax = top_countries["works"].max()
for bar, val in zip(bars, top_countries["works"]):
    ax_l.text(bar.get_width() + xmax * 0.01, bar.get_y() + bar.get_height() / 2,
              f"{int(val):,}", va="center", ha="left", fontsize=9, color="#333")
ax_l.set_xlim(0, xmax * 1.12)

# Right: works per year — filled area + line + markers
accent = "#2a6f97"
ax_r.fill_between(years.index, years.values, color=accent, alpha=0.18)
ax_r.plot(years.index, years.values, color=accent, linewidth=2.2,
          marker="o", markersize=5, markerfacecolor="white",
          markeredgecolor=accent, markeredgewidth=1.6)
ax_r.set_title("Works per publication year", loc="left", pad=12)
ax_r.set_xlabel("year"); ax_r.set_ylabel("works")
ax_r.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax_r.grid(axis="y", linestyle=":", alpha=0.5)
ax_r.set_axisbelow(True)

# Highlight peak year
peak_year = int(years.idxmax()); peak_val = int(years.max())
ax_r.annotate(f"peak: {peak_val:,} ({peak_year})",
              xy=(peak_year, peak_val),
              xytext=(0, 18), textcoords="offset points",
              ha="center", fontsize=9, color=accent, fontweight="semibold",
              arrowprops=dict(arrowstyle="-", color=accent, lw=0.8))

#fig.suptitle("Where and when: first-author geography & publication volume",
#             x=0.012, y=0.99, ha="left", fontsize=15, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 3. Term-frequency in abstracts, by year (grouped)

For each abstract we count non-overlapping hits per AI group from `AI_GROUPS`,
plus comparators (`machine learning`, `deep learning`).

**Overlap policy.** A sentence like *"using generative AI and an AI agent"*
yields `generative_ai=1, agent=1, general_ai=0`. The bare `general_ai` bucket
only catches free-standing "AI" / "artificial intelligence" / "AGI" tokens
that are not absorbed by a more specific group.

In [ ]:
GROUP_COLS = [f"n_{g}" for g in AI_GROUPS]
cmp_re = compile_comparator_regex()

import re as _re
_ML_RE = _re.compile(r"\bmachine\s+learning\b", flags=_re.IGNORECASE)
_DL_RE = _re.compile(r"\bdeep\s+learning\b", flags=_re.IGNORECASE)
_TRANS_RE = _re.compile(r"\btransformer\b", flags=_re.IGNORECASE)


def _mention_row(text: str | None) -> dict:
    counts = classify_text(text)
    row = {f"n_{g}": counts[g] for g in AI_GROUPS}
    row["n_ai_total"] = sum(counts.values())
    row["has_ai"] = row["n_ai_total"] > 0
    if text:
        row["n_ml"] = len(_ML_RE.findall(text))
        row["n_dl"] = len(_DL_RE.findall(text))
        row["n_trans"] = len(_TRANS_RE.findall(text))
    else:
        row["n_ml"] = row["n_dl"] = row["n_trans"] = 0
    return row


flags = pd.DataFrame(df["abstract"].map(_mention_row).tolist())
df_terms = pd.concat([df[["id", "publication_year"]], flags], axis=1)
df_terms.head()


In [ ]:
# Per-year aggregation: count of works whose abstract contains >=1 hit in each bucket.
has_cols = {f"has_{g}": (df_terms[f"n_{g}"] > 0) for g in AI_GROUPS}
has_df = pd.DataFrame(has_cols)
df_terms_b = pd.concat([df_terms, has_df], axis=1)

agg = {"n_works": ("id", "count"), "n_ai": ("has_ai", "sum"),
       "n_ml": ("n_ml", "sum"), "n_dl": ("n_dl", "sum"), "n_trans": ("n_trans", "sum")}
for g in AI_GROUPS:
    agg[f"n_{g}"] = (f"has_{g}", "sum")           # works whose abstract has >=1
    agg[f"hits_{g}"] = (f"n_{g}", "sum")          # total hit count

by_year = (
    df_terms_b.dropna(subset=["publication_year"])
    .assign(publication_year=lambda d: d["publication_year"].astype(int))
    .groupby("publication_year").agg(**agg).reset_index()
)
for g in AI_GROUPS:
    by_year[f"share_{g}"] = by_year[f"n_{g}"] / by_year["n_works"]
by_year["share_ai"] = by_year["n_ai"] / by_year["n_works"]

out_terms = OUT_MACRO / "term_counts_by_year.parquet"
by_year.to_parquet(out_terms, index=False)
by_year.tail(10)


In [ ]:
by_year.keys()

### 3.1 Mentions per group over time (line plot)

The primary diachronic view: total **hit count** per AI group per publication
year. Each group has a distinct colour; the encoder/decoder/image-gen groups
plus `generative_ai`, `llm`, `chatbot`, `agent`, `general_ai` are all on the
same axes so their relative weight is directly comparable.

We show two panels:
* **all years** with a log y-axis (handles the sparse pre-2018 tail and the
  2024-2025 spike on the same chart);
* **2015 onward** with a linear y-axis to read off the recent absolute volume.


In [ ]:
# Distinct color + marker per group (8 groups -> tab10 cycle).
GROUP_LIST = list(AI_GROUPS)
_PALETTE = plt.get_cmap("tab10").colors
_MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
GROUP_STYLE = {g: {"color": _PALETTE[i % 10], "marker": _MARKERS[i % len(_MARKERS)]}
               for i, g in enumerate(GROUP_LIST)}


def _plot_group_lines(ax, sub, ycol_fmt="hits_{}", **kwargs):
    for g in GROUP_LIST:
        ax.plot(sub["publication_year"], sub[ycol_fmt.format(g)],
                label=g, linewidth=1.6, markersize=4.5,
                **GROUP_STYLE[g], **kwargs)


fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

sub_all = by_year[by_year["publication_year"] >= 1990].sort_values("publication_year")
_plot_group_lines(axes[0], sub_all)
axes[0].set_yscale("log")
axes[0].set_title("AI-group mentions in abstracts per year (all years, log scale)")
axes[0].set_xlabel("publication year"); axes[0].set_ylabel("mentions (hits, log)")
axes[0].grid(True, which="both", linestyle=":", alpha=0.4)
axes[0].legend(fontsize=8, loc="upper left", ncol=2, frameon=True)

sub_recent = by_year[by_year["publication_year"] >= 2015].sort_values("publication_year")
_plot_group_lines(axes[1], sub_recent)
axes[1].set_title("AI-group mentions in abstracts per year (2015+, linear)")
axes[1].set_xlabel("publication year"); axes[1].set_ylabel("mentions (hits)")
axes[1].grid(True, linestyle=":", alpha=0.4)
axes[1].legend(fontsize=8, loc="upper left", ncol=2, frameon=True)

plt.tight_layout(); plt.show()


### 3.2 Share of works per group + ML/DL comparators (percentages)

Same groups, but normalised to the per-year work count (so the y-axis is a
**percentage of the works published that year** that contain at least one
hit from the group). The `machine learning` / `deep learning` / `transformers` comparator
curves are overlaid in black.

Two panels:
* **all years**, linear y-axis as %;
* **2015+ zoom**, linear y-axis as %, to read off the recent surge.

In [ ]:
from matplotlib.ticker import PercentFormatter

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

sub_all = by_year[by_year["publication_year"] >= 1990].sort_values("publication_year")
_plot_group_lines(axes[0], sub_all, ycol_fmt="share_{}")
axes[0].plot(sub_all["publication_year"], sub_all["n_ml"] / sub_all["n_works"],
             marker="x", linestyle="--", color="black", linewidth=1.2,
             label="machine learning")
axes[0].plot(sub_all["publication_year"], sub_all["n_dl"] / sub_all["n_works"],
             marker="x", linestyle=":", color="dimgray", linewidth=1.2,
             label="deep learning")
axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=1))
axes[0].set_title("Share of works per year with >=1 hit, by group (all years)")
axes[0].set_xlabel("publication year"); axes[0].set_ylabel("share of works")
axes[0].grid(True, linestyle=":", alpha=0.4)
axes[0].legend(fontsize=8, loc="upper left", ncol=2, frameon=True)

sub_recent = by_year[by_year["publication_year"] >= 2015].sort_values("publication_year")
_plot_group_lines(axes[1], sub_recent, ycol_fmt="share_{}")
axes[1].plot(sub_recent["publication_year"], sub_recent["n_ml"] / sub_recent["n_works"],
             marker="x", linestyle="--", color="black", linewidth=1.2,
             label="machine learning")
axes[1].plot(sub_recent["publication_year"], sub_recent["n_dl"] / sub_recent["n_works"],
             marker="x", linestyle=":", color="dimgray", linewidth=1.2,
             label="deep learning")
axes[1].plot(sub_recent["publication_year"], sub_recent["n_trans"] / sub_recent["n_works"],
             marker="x", linestyle="-.", color="gray", linewidth=1.2,
             label="transformer")
axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=1))
axes[1].set_title("Share of works per year with >=1 hit, by group (2015+)")
axes[1].set_xlabel("publication year"); axes[1].set_ylabel("share of works")
axes[1].grid(True, linestyle=":", alpha=0.4)
axes[1].legend(fontsize=8, loc="upper left", ncol=2, frameon=True)

plt.tight_layout(); plt.show()


In [ ]:
#calculate number of papers mentioning AI per year
ai_mentions_per_year = df_terms.groupby("publication_year")["has_ai"].sum()
#calculate total number of papers per year
total_papers_per_year = df_terms.groupby("publication_year")["id"].count()

### 3.3 Composition of mentions (stacked, 2015+)

Stacked-bar view: of all AI-group mentions in a given year, what fraction is
contributed by each group? Useful for spotting the 2023+ shift from
encoder-style transformers (BERT/T5) toward decoder-only LLMs and
`generative_ai`/`llm` vocabulary.

In [ ]:
recent = by_year[by_year["publication_year"] >= 2018].set_index("publication_year")
stack_cols = [f"hits_{g}" for g in GROUP_LIST]
colors = [GROUP_STYLE[g]["color"] for g in GROUP_LIST]
# Normalise hits by the number of works published that year so bar heights
# reflect mention intensity per work, not raw corpus growth.
recent_norm = recent[stack_cols].div(recent["n_works"], axis=0) * 1000
ax = recent_norm.plot(kind="bar", stacked=True, figsize=(13, 4.5),
                      color=colors)
ax.set_title("AI-term hits per 1000 abstracts per year, by group (stacked)")
ax.set_ylabel("hits per 1000 abstracts"); ax.set_xlabel("year")
ax.legend([c.replace("hits_", "") for c in stack_cols],
          fontsize=8, loc="upper left", ncol=2)
plt.tight_layout(); plt.show()


### 3.4 Do "AI" mentions co-occur with named models?

Question: when an abstract uses *general* AI vocabulary
(`general_ai` — bare "AI" / "artificial intelligence" / "AGI —
or `generative_ai` — "generative AI" / "GenAI"), how often does it also
name a concrete model family (`named_encoder_decoder`, `named_decoder_only`,
`named_image_generator`)?

A high share would mean the generic AI vocabulary is anchored to specific
systems; a low share means "AI" is used as a free-floating label without
naming what is actually being used.

In [ ]:
GEN_GROUPS   = ["general_ai", "generative_ai"]
NAMED_GROUPS = ["named_encoder_decoder", "named_decoder_only", "named_image_generator"]

# Per-abstract boolean flags.
has_gen   = (df_terms[[f"n_{g}" for g in GEN_GROUPS]].sum(axis=1) > 0)
has_named = {g: (df_terms[f"n_{g}"] > 0) for g in NAMED_GROUPS}
has_any_named = pd.concat(has_named.values(), axis=1).any(axis=1)

# Overall (corpus-wide) co-occurrence table.
overall = pd.DataFrame({
    "abstracts_with_general/generative_AI": [int(has_gen.sum())],
    "  also naming a model (any)":          [int((has_gen & has_any_named).sum())],
    "  also naming encoder_decoder":        [int((has_gen & has_named["named_encoder_decoder"]).sum())],
    "  also naming decoder_only":           [int((has_gen & has_named["named_decoder_only"]).sum())],
    "  also naming image_generator":        [int((has_gen & has_named["named_image_generator"]).sum())],
}).T.rename(columns={0: "n"})
overall["share_of_general"] = overall["n"] / max(int(has_gen.sum()), 1)
overall.loc["abstracts_with_general/generative_AI", "share_of_general"] = 1.0
print(overall.round(3))

# Reverse direction: of abstracts naming a model, how many also use "AI" vocab?
rev = pd.DataFrame({
    "abstracts_naming_any_model": [int(has_any_named.sum())],
    "  also using general/generative AI": [int((has_any_named & has_gen).sum())],
}).T.rename(columns={0: "n"})
rev["share_of_named"] = rev["n"] / max(int(has_any_named.sum()), 1)
rev.loc["abstracts_naming_any_model", "share_of_named"] = 1.0
print(rev.round(3))


In [ ]:
from collections import Counter, defaultdict
import re as _re_fam
import regex as _re_named
import numpy as np

# ------------------------------------------------------------------
# (A) Build sub + _per_year_models EXPLICITLY from df + df_terms.
#
# Scope: only abstracts that also use general/generative-AI vocab
# (matches the 3.4 question: "when an abstract says AI, does it also
# name a model?"). For each such abstract, we collect the literal
# match strings of the named_* groups; counts are aggregated per year.
# ------------------------------------------------------------------
NAMED_GROUPS = ["named_encoder_decoder", "named_decoder_only", "named_image_generator"]
GEN_GROUPS   = ["general_ai", "generative_ai"]

# Combined regex over JUST the named groups (case-insensitive, like the lexicon).
_named_pat = _re_named.compile(
    "|".join(p for g in NAMED_GROUPS for p in AI_GROUPS[g]),
    flags=_re_named.IGNORECASE | _re_named.UNICODE,
)

_has_gen   = (df_terms[[f"n_{g}" for g in GEN_GROUPS]].sum(axis=1) > 0).values
_has_named = pd.concat(
    [(df_terms[f"n_{g}"] > 0) for g in NAMED_GROUPS], axis=1
).any(axis=1).values
_years_arr = df["publication_year"].values
_abstracts = df["abstract"].values
_n_per_g   = {g: df_terms[f"n_{g}"].values for g in NAMED_GROUPS}

_per_year_models: dict[int, Counter] = defaultdict(Counter)
_n_general    = defaultdict(int)
_n_any_named  = defaultdict(int)
_n_per_named  = {g: defaultdict(int) for g in NAMED_GROUPS}

for i in range(len(df)):
    if not _has_gen[i]:
        continue
    y = _years_arr[i]
    if y is None or (isinstance(y, float) and np.isnan(y)):
        continue
    y = int(y)
    _n_general[y] += 1
    if _has_named[i]:
        _n_any_named[y] += 1
    for g in NAMED_GROUPS:
        if _n_per_g[g][i] > 0:
            _n_per_named[g][y] += 1
    abstract = _abstracts[i]
    if abstract:
        for m in _named_pat.finditer(abstract):
            _per_year_models[y][m.group(0)] += 1

_years_all = sorted(_n_general.keys())
sub = pd.DataFrame({
    "publication_year": _years_all,
    "n_general":        [_n_general[y]   for y in _years_all],
    "n_any_named":      [_n_any_named[y] for y in _years_all],
    **{f"n_{g}": [_n_per_named[g][y] for y in _years_all] for g in NAMED_GROUPS},
})
sub["share_any_named"] = sub["n_any_named"] / sub["n_general"].clip(lower=1)
for g in NAMED_GROUPS:
    sub[f"share_{g}"] = sub[f"n_{g}"] / sub["n_general"].clip(lower=1)

# Restrict to years with at least a handful of general/generative-AI abstracts.
sub = sub[(sub["publication_year"] >= 2023) & (sub["n_general"] >= 5)].reset_index(drop=True)

# ------------------------------------------------------------------
# (B) Collapse model variants into a single family bucket so the heatmap
# shows e.g. one "GPT" row instead of GPT / GPT-3 / GPT-4 / ChatGPT.
# Patterns are tried in order; first match wins.
# ------------------------------------------------------------------
_FAMILY_RULES = [
    # decoder-only LLMs
    ("GPT",              r"^(?:chat)?gpt\b"),
    ("Claude",           r"^claude\b"),
    ("Gemini",           r"^gemini\b"),
    ("Bard",             r"^bard\b"),
    ("Gemma",            r"^gemma\b"),
    ("LLaMA",            r"^l?lama\b|^llama\b"),
    ("Mistral",          r"^mi[xs]tral\b"),
    ("Copilot",          r"^(?:github\s+)?copilot\b"),
    ("Phi",              r"^phi\b"),
    ("Grok",             r"^grok\b"),
    ("Command",          r"^command\b"),
    ("Aya",              r"^(?:tiny[-\s]?)?aya\b"),
    ("DeepSeek",         r"^deep[\s-]?seek\b"),
    ("Falcon",           r"^falcon\b"),
    ("GLM",              r"^glm\b"),
    ("Granite",          r"^granite\b"),
    ("Kimi",             r"^kimi\b"),
    ("MiniMax",          r"^minimax\b"),
    ("Nemotron",         r"^nemotron\b"),
    ("Qwen",             r"^qwen\b"),
    ("Yi",               r"^yi\b"),
    ("OLMo",             r"^olmo\b|^allenai\b"),
    # encoder / encoder-decoder
    ("BERT",             r"bert\b"),                # RoBERTa, DistilBERT, ALBERT, DeBERTa, SciBERT...
    ("ELECTRA",          r"^electra\b"),
    ("XLNet",            r"^xlnet\b"),
    ("XLM-R",            r"^xlm"),
    ("T5",               r"t5\b"),                  # T5, mT5, FLAN-T5, ByT5
    ("BART",             r"^m?bart\b"),
    ("Pegasus",          r"^pegasus\b"),
    # image / multimodal generators
    ("DALL-E",           r"^dall[-\u00b7.\s]?e\b|^dalle\b"),
    ("Midjourney",       r"^midjourney\b"),
    ("Stable Diffusion", r"^stable\s+diffusion\b"),
    ("Imagen",           r"^imagen\b"),
    ("Sora",             r"^sora\b"),
]
_FAMILY_RES = [(fam, _re_fam.compile(pat, _re_fam.IGNORECASE)) for fam, pat in _FAMILY_RULES]


def _canon_family(name: str) -> str | None:
    s = name.strip()
    if not s:
        return None
    for fam, rx in _FAMILY_RES:
        if rx.search(s):
            return fam
    return s  # unknown -> keep as-is

# Rebuild per-year Counters keyed by family (collapse variants).
_per_year_models = {
    y: Counter({
        f: sum(c for m, c in orig_cnt.items() if _canon_family(m) == f)
        for f in {_canon_family(m) for m in orig_cnt if _canon_family(m)}
    })
    for y, orig_cnt in _per_year_models.items()
}

# Drop anachronistic matches: a model can't be mentioned before it existed.
# (e.g. "Claude" pre-2023 is almost certainly a person's name.)
_MIN_YEAR = {
    "Claude": 2023,
    "Gemini": 2023
}
for y, cnt in _per_year_models.items():
    for fam, min_y in _MIN_YEAR.items():
        if y < min_y and fam in cnt:
            del cnt[fam]

top_model_per_year = {
    y: (c.most_common(1)[0] if c else (None, 0))
    for y, c in _per_year_models.items()
}

# ------------------------------------------------------------------
# Two-panel figure:
#   (top)    share lines per named_* group + "any named" — no annotations
#   (bottom) heatmap of top-K model families x year (colour = mentions)
# Cleaner than annotating every point: the heatmap shows not just the
# winner but also the runners-up and the relative gap.
# ------------------------------------------------------------------
TOP_K_MODELS = 10
# Drop noisy/ambiguous family names (e.g. "Flux" matches the JS library;
# "PaLM" is rare in this corpus and mostly false positives).
EXCLUDE_MODELS = {"Flux", "PaLM"}
_excl_lower = {m.lower() for m in EXCLUDE_MODELS}
all_counts: Counter = Counter()
for c in _per_year_models.values():
    all_counts.update(c)
top_models = [m for m, _ in all_counts.most_common()
              if m.lower() not in _excl_lower][:TOP_K_MODELS]

years_sorted = sorted(sub["publication_year"].astype(int).tolist())
heat = pd.DataFrame(
    {y: [_per_year_models.get(y, {}).get(m, 0) for m in top_models] for y in years_sorted},
    index=top_models,
)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(12, 8.5),
    gridspec_kw={"height_ratios": [1.0, 1.1], "hspace": 0.35},
)

# --- Top: share lines ---
for g in NAMED_GROUPS:
    ax_top.plot(sub["publication_year"], sub[f"share_{g}"],
                label=g, linewidth=1.6, markersize=4.5, **GROUP_STYLE[g])
ax_top.plot(sub["publication_year"], sub["share_any_named"],
            label="any named model", linewidth=2.0, color="black", marker="o")
ax_top.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax_top.set_title("Among abstracts with general/generative AI vocab:\n"
                 "share also naming a model")
ax_top.set_ylabel("share of general/generative-AI abstracts")
ax_top.grid(True, linestyle=":", alpha=0.4)
ax_top.legend(fontsize=8, loc="upper left", frameon=True)
ax_top.set_xticks(sub["publication_year"])
ax_top.set_xticklabels(
    [f"{y}\n(n={n})" for y, n in zip(sub["publication_year"], sub["n_general"])],
    fontsize=8,
)
ax_top.set_ylim(0, max(sub["share_any_named"].max() * 1.15, 0.25))

# --- Bottom: heatmap of top-K models per year ---
import numpy as np
data = heat.values.astype(float)
im = ax_bot.imshow(np.log1p(data), aspect="auto", cmap="magma")
ax_bot.set_yticks(range(len(top_models)))
ax_bot.set_yticklabels(top_models, fontsize=9)
ax_bot.set_xticks(range(len(years_sorted)))
ax_bot.set_xticklabels(years_sorted, fontsize=9)
ax_bot.set_title(f"Top {TOP_K_MODELS} named model families — mentions per year "
                 "(log-scaled colour, raw counts in cells)")
ax_bot.set_xlabel("publication year")

# Align grid with cell boundaries: drop the seaborn major grid and draw
# minor-tick lines at the half-integer offsets that bound each imshow cell.
ax_bot.grid(False)
ax_bot.set_xticks(np.arange(-0.5, len(years_sorted), 1), minor=True)
ax_bot.set_yticks(np.arange(-0.5, len(top_models), 1), minor=True)
ax_bot.grid(which="minor", color="white", linestyle="-", linewidth=0.6)
ax_bot.tick_params(which="minor", length=0)

# Write raw count in each non-zero cell.
_vmax_log = max(np.log1p(data.max()), 1e-9)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        v = int(data[i, j])
        if v == 0:
            continue
        norm = np.log1p(v) / _vmax_log
        color = "white" if norm < 0.55 else "black"
        ax_bot.text(j, i, str(v), ha="center", va="center",
                    fontsize=7.5, color=color)

cbar = fig.colorbar(im, ax=ax_bot, fraction=0.025, pad=0.01)
cbar.set_label("log(1 + mentions)")

plt.show()

### 3.4b Named model mentions per year (all abstracts)

Same heatmap as 3.4, but scoped to **every** abstract — not only those that
also use general/generative-AI vocabulary. This shows the raw diachronic
footprint of named model families in the corpus.

In [ ]:
# Reuse the named-model regex + family canonicaliser from cell 3.4.
# Scope: all abstracts (no _has_gen filter), restricted to 2023+.

_per_year_models_all: dict[int, Counter] = defaultdict(Counter)
_n_works_year: dict[int, int] = defaultdict(int)

for i in range(len(df)):
    y = _years_arr[i]
    if y is None or (isinstance(y, float) and np.isnan(y)):
        continue
    y = int(y)
    _n_works_year[y] += 1
    abstract = _abstracts[i]
    if not abstract:
        continue
    for m in _named_pat.finditer(abstract):
        _per_year_models_all[y][m.group(0)] += 1

# Collapse variants to families.
_per_year_models_all = {
    y: Counter({
        f: sum(c for m, c in orig_cnt.items() if _canon_family(m) == f)
        for f in {_canon_family(m) for m in orig_cnt if _canon_family(m)}
    })
    for y, orig_cnt in _per_year_models_all.items()
}

# Anachronism filter (same rule as 3.4).
for y, cnt in _per_year_models_all.items():
    for fam, min_y in _MIN_YEAR.items():
        if y < min_y and fam in cnt:
            del cnt[fam]

# Top-K families across the (>=2023) window.
years_all = sorted(y for y in _per_year_models_all.keys() if y >= 2023)
all_counts_all: Counter = Counter()
for y in years_all:
    all_counts_all.update(_per_year_models_all[y])
top_models_all = [m for m, _ in all_counts_all.most_common()
                  if m.lower() not in _excl_lower][:TOP_K_MODELS]

heat_all = pd.DataFrame(
    {y: [_per_year_models_all.get(y, {}).get(m, 0) for m in top_models_all] for y in years_all},
    index=top_models_all,
)

fig, ax_bot = plt.subplots(figsize=(12, 5.0))
data = heat_all.values.astype(float)
im = ax_bot.imshow(np.log1p(data), aspect="auto", cmap="magma")
ax_bot.set_yticks(range(len(top_models_all)))
ax_bot.set_yticklabels(top_models_all, fontsize=9)
ax_bot.set_xticks(range(len(years_all)))
ax_bot.set_xticklabels(
    [f"{y}\n(N={_n_works_year[y]})" for y in years_all], fontsize=9,
)
ax_bot.set_title(f"Top {TOP_K_MODELS} named model families — mentions per year "
                 "(all abstracts, log-scaled colour, raw counts in cells)")
ax_bot.set_xlabel("publication year")

ax_bot.grid(False)
ax_bot.set_xticks(np.arange(-0.5, len(years_all), 1), minor=True)
ax_bot.set_yticks(np.arange(-0.5, len(top_models_all), 1), minor=True)
ax_bot.grid(which="minor", color="white", linestyle="-", linewidth=0.6)
ax_bot.tick_params(which="minor", length=0)

_vmax_log = max(np.log1p(data.max()), 1e-9)
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        v = int(data[i, j])
        if v == 0:
            continue
        norm = np.log1p(v) / _vmax_log
        color = "white" if norm < 0.55 else "black"
        ax_bot.text(j, i, str(v), ha="center", va="center",
                    fontsize=7.5, color=color)

cbar = fig.colorbar(im, ax=ax_bot, fraction=0.025, pad=0.01)
cbar.set_label("log(1 + mentions)")
plt.tight_layout()
plt.show()

## 4. Headline numbers

A small summary printed for the write-up.

In [ ]:
summary = {
    "total_works": int(len(df)),
    "with_abstract": int(df["abstract"].notna().sum()),
    "share_with_abstract": round(df["abstract"].notna().mean(), 3),
    "oa_share": round(df["is_oa"].mean(), 3) if df["is_oa"].notna().any() else None,
    "english_share": round((df["language"] == "en").mean(), 3),
    "year_min": int(df["publication_year"].min()),
    "year_max": int(df["publication_year"].max()),
    "ai_mentioning_abstracts": int(df_terms["has_ai"].sum()),
    "ml_mentioning_abstracts": int((df_terms["n_ml"] > 0).sum()),
    "dl_mentioning_abstracts": int((df_terms["n_dl"] > 0).sum()),
    "by_group_works": {g: int((df_terms[f"n_{g}"] > 0).sum()) for g in AI_GROUPS},
    "by_group_hits":  {g: int(df_terms[f"n_{g}"].sum()) for g in AI_GROUPS},
}
summary
